# Tabular Data Processing
To get the data Genie-ready we have to perform transformations on the Excel files

### 📦 Setup

In [0]:
%pip install openpyxl
%restart_python

In [0]:
import pandas as pd
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="openpyxl"
)

### 2024 Tables
These tables include information for the whole 2023 and 2024 years.

In [0]:
file_path = "/Volumes/sds/raw/tabular/Financial Tables FY24.xlsx"
excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

In [0]:
df = pd.read_excel(file_path, sheet_name="Income Statement")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
df = df.drop(df.columns[1], axis=1) # drop the note column

# 2. Rename columns
df.columns = ['identity', 'fy_2024', 'fy_2023'] + [f'extra_{i}' for i in range(4, len(df.columns) + 1)]
df = df[['identity', 'fy_2024', 'fy_2023']].reset_index(drop=True)

# 3. Find the "In millions of CHF" row and drop everything above it
# We find the index of the first row containing 'Net sales' (the first real data point)
start_index = df[df['identity'].str.contains("Net sales", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Contextualize Labels (Prepend "ATTRIBUTABLE TO" and "EARNINGS PER SHARE")
current_context = ""
clean_identities = []

for idx, row in df.iterrows():
    name = str(row['identity']).strip()
    val_2024 = str(row['fy_2024']).lower()
    
    # If the row has text but NO numbers (NaN), it's a context header
    if val_2024 == 'nan' or val_2024 == '':
        current_context = name
        clean_identities.append(name) # Keep it for now, we drop it later
    else:
        # If there is a context active, prepend it
        if current_context:
            clean_identities.append(f"{current_context}: {name}")
        else:
            clean_identities.append(name)

df['identity'] = clean_identities

# 5. Numeric Normalization & Dropping Empty Rows
# Convert columns to numeric, forcing errors (like 'NaN' strings) to actual NaNs
df['fy_2024'] = pd.to_numeric(df['fy_2024'], errors='coerce')
df['fy_2023'] = pd.to_numeric(df['fy_2023'], errors='coerce')

# Drop the rows that are purely headers (where fy_2024 is now NaN)
df_2024_income_statement = df.dropna(subset=['fy_2024']).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="Balance sheet")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
df = df.drop(df.columns[1], axis=1) # drop the note column

# 2. Rename columns
df.columns = ['identity', 'fy_2024', 'fy_2023'] + [f'extra_{i}' for i in range(4, len(df.columns) + 1)]
df = df[['identity', 'fy_2024', 'fy_2023']].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("Property, plant, and equipment", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['fy_2024'] = pd.to_numeric(df['fy_2024'], errors='coerce')
df['fy_2023'] = pd.to_numeric(df['fy_2023'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2024_balance_sheet = df.dropna(subset=['fy_2024', 'fy_2023'], how='all').reset_index(drop=True).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="Cash Flow")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
df = df.drop(df.columns[1], axis=1) # drop the note column

# 2. Rename columns
df.columns = ['identity', 'fy_2024', 'fy_2023'] + [f'extra_{i}' for i in range(4, len(df.columns) + 1)]
df = df[['identity', 'fy_2024', 'fy_2023']].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("Profit before tax", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['fy_2024'] = pd.to_numeric(df['fy_2024'], errors='coerce')
df['fy_2023'] = pd.to_numeric(df['fy_2023'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2024_cash_flow = df.dropna(subset=['fy_2024', 'fy_2023'], how='all').reset_index(drop=True).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name=" Core and IFRS profit or loss")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')

# 2. Rename columns
cols_rename = ['identity', 'core_2024_percent', 'core_2024', 'ifrs_2024_percent', 'ifrs_2024', 'ifrs_2023']
df.columns = cols_rename + [f'extra_{i}' for i in range(7, len(df.columns) + 1)]
df = df[cols_rename].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("Turnover", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['core_2024_percent'] = pd.to_numeric(df['core_2024_percent'], errors='coerce')
df['core_2024'] = pd.to_numeric(df['core_2024'], errors='coerce')
df['ifrs_2024_percent'] = pd.to_numeric(df['ifrs_2024_percent'], errors='coerce')
df['ifrs_2024'] = pd.to_numeric(df['ifrs_2024'], errors='coerce')
df['ifrs_2023'] = pd.to_numeric(df['ifrs_2023'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2024_core_ifrs = df.dropna(subset=['core_2024', 'ifrs_2024', 'ifrs_2023'], how='all').reset_index(drop=True).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="Equity free cash flow")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')

# 2. Rename columns
df.columns = ['identity', 'fy_2024', 'fy_2024_percentage', 'fy_2023', 'fy_2023_percentage'] + [f'extra_{i}' for i in range(6, len(df.columns) + 1)]
df = df[['identity', 'fy_2024', 'fy_2024_percentage', 'fy_2023', 'fy_2023_percentage']].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("CORE EBITDA", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['fy_2024'] = pd.to_numeric(df['fy_2024'], errors='coerce')
df['fy_2023'] = pd.to_numeric(df['fy_2023'], errors='coerce')
df['fy_2024_percentage'] = pd.to_numeric(df['fy_2024_percentage'], errors='coerce')
df['fy_2023_percentage'] = pd.to_numeric(df['fy_2023_percentage'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2024_equity_free_cash_flow = df.dropna(subset=['fy_2024', 'fy_2023'], how='all').reset_index(drop=True).copy()

### 2025 Tables
These tables include information for the first 6 months of 2024 and 2025

In [0]:
file_path = "/Volumes/sds/raw/tabular/Financial Tables HY2025_0.xlsx"
excel_file = pd.ExcelFile(file_path)

print(excel_file.sheet_names)

In [0]:
df = pd.read_excel(file_path, sheet_name="Income Statement")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
df = df.drop(df.columns[1], axis=1) # drop the note column

# 2. Rename columns
df.columns = ['identity', '6m_2025', '6m_2024'] + [f'extra_{i}' for i in range(4, len(df.columns) + 1)]
df = df[['identity', '6m_2025', '6m_2024']].reset_index(drop=True)

# 3. Find the start of the data
# We look for 'Net sales' to skip the "CONSOLIDATED", "UNAUDITED", etc.
try:
    start_index = df[df['identity'].str.contains("Net sales", na=False)].index[0]
    df = df.iloc[start_index:].reset_index(drop=True)
except IndexError:
    print("Warning: 'Net sales' not found. Check if the column name is correct.")

# 4. Contextualize Labels (Prepend "Attributable to" etc.)
current_context = ""
clean_identities = []

for idx, row in df.iterrows():
    name = str(row['identity']).strip()
    val_2025 = str(row['6m_2025']).lower()
    
    # Identify Header Rows (Text present, but 2025 value is NaN/Empty)
    if val_2025 == 'nan' or val_2025 == '':
        current_context = name
        clean_identities.append(name)
    else:
        # Prepend the context (e.g., "Attributable to: Non-controlling interests")
        if current_context and current_context.lower() != 'nan':
            clean_identities.append(f"{current_context}: {name}")
        else:
            clean_identities.append(name)

df['identity'] = clean_identities

# 5. Numeric Normalization
df['6m_2025'] = pd.to_numeric(df['6m_2025'], errors='coerce')
df['6m_2024'] = pd.to_numeric(df['6m_2024'], errors='coerce')

# Drop the header rows that now have NaN in the numeric column
df_2025_income_statement = df.dropna(subset=['6m_2025']).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="Balance sheet")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
df = df.drop(df.columns[1], axis=1) # drop the note column

# 2. Rename columns
df.columns = ['identity', '6m_2025', 'fy_2024'] + [f'extra_{i}' for i in range(4, len(df.columns) + 1)]
df = df[['identity', '6m_2025', 'fy_2024']].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("Property, plant and equipment", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['6m_2025'] = pd.to_numeric(df['6m_2025'], errors='coerce')
df['fy_2024'] = pd.to_numeric(df['fy_2024'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2025_balance_sheet = df.dropna(subset=['6m_2025', 'fy_2024'], how='all').reset_index(drop=True).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="Cash Flow")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
df = df.drop(df.columns[1], axis=1) # drop the note column

# 2. Rename columns
df.columns = ['identity', '6m_2025', '6m_2024'] + [f'extra_{i}' for i in range(4, len(df.columns) + 1)]
df = df[['identity', '6m_2025', '6m_2024']].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("Profit before tax", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['6m_2025'] = pd.to_numeric(df['6m_2025'], errors='coerce')
df['6m_2024'] = pd.to_numeric(df['6m_2024'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2025_cash_flow = df.dropna(subset=['6m_2025', '6m_2024'], how='all').reset_index(drop=True).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="CORE P&L ")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')

# 2. Rename columns
cols_rename = ['identity', '6m_2025', '6m_2025_percent', '6m_2024', '6m_2024_percent']
df.columns = cols_rename + [f'extra_{i}' for i in range(6, len(df.columns) + 1)]
df = df[cols_rename].reset_index(drop=True)

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("In millions of CHF", na=False)].index[0]
df = df.iloc[1:].reset_index(drop=True)
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['6m_2025'] = pd.to_numeric(df['6m_2025'], errors='coerce')
df['6m_2025_percent'] = pd.to_numeric(df['6m_2025_percent'], errors='coerce')
df['6m_2024'] = pd.to_numeric(df['6m_2024'], errors='coerce')
df['6m_2024_percent'] = pd.to_numeric(df['6m_2024_percent'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2025_core_ifrs = df.dropna(subset=['6m_2025', '6m_2024'], how='all').reset_index(drop=True).copy()

In [0]:
df = pd.read_excel(file_path, sheet_name="CORE Cash Flow")

# 1. Drop rows and columns where all values are null
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')

# 2. Rename columns
df.columns = ['identity', '6m_2025', '6m_2025_percentage', '6m_2024', '6m_2024_percentage'] + [f'extra_{i}' for i in range(6, len(df.columns) + 1)]
df = df[['identity', '6m_2025', '6m_2025_percentage', '6m_2024', '6m_2024_percentage']].reset_index(drop=True) 

# 3. We find the index of the first row containing real data (the first real data point)
start_index = df[df['identity'].str.contains("CORE EBITDA", na=False)].index[0]
df = df.iloc[start_index:].reset_index(drop=True)

# 4. Numeric Normalization & Dropping Empty Rows
df['6m_2025'] = pd.to_numeric(df['6m_2025'], errors='coerce')
df['6m_2024'] = pd.to_numeric(df['6m_2024'], errors='coerce')
df['6m_2025_percentage'] = pd.to_numeric(df['6m_2025_percentage'], errors='coerce')
df['6m_2024_percentage'] = pd.to_numeric(df['6m_2024_percentage'], errors='coerce')

# 5. Drop every row that doesn't have any numeric values
df_2025_equity_free_cash_flow = df.dropna(subset=['6m_2025', '6m_2024'], how='all').reset_index(drop=True).copy()

### 💾 Saving Tables to Unity Catalog

Now we take all processed tables and deploy them.

In [ ]:
%sql
USE CATALOG sds;

In [0]:
# 1. Define the mapping of your DataFrames to the NotebookLM descriptions
# We will save these to the 'sds.cleaned' schema
tables_to_deploy = {
    "income_statement_2024": {
        "df": df_2024_income_statement,
        "comment": "Consolidated Statement of Profit or Loss (FY 2024/2023): Details total revenue, cost of sales, operating expenses, finance costs, and final net profit/loss."
    },
    "balance_sheet_2024": {
        "df": df_2024_balance_sheet,
        "comment": "Consolidated Statement of Financial Position (FY 2024/2023): A snapshot of Avolta's assets, liabilities, and equity at end of 2024."
    },
    "cash_flow_2024": {
        "df": df_2024_cash_flow,
        "comment": "Consolidated Statement of Cash Flows (FY 2024/2023): Breakdown of cash from operating, investing, and financing activities."
    },
    "core_ifrs_2024": {
        "df": df_2024_core_ifrs,
        "comment": "CORE and IFRS Profit or Loss (FY 2024/2023): Side-by-side comparison of statutory IFRS vs Avolta's CORE management metrics."
    },
    "equity_free_cash_flow_2024": {
        "df": df_2024_equity_free_cash_flow,
        "comment": "Equity Free Cash Flow (EFCF) (FY 2024/2023): Calculation of cash available to shareholders, starting from CORE EBITDA."
    },
    "income_statement_2025_6m": {
        "df": df_2025_income_statement,
        "comment": "Consolidated Statement of Profit or Loss (6M 2025/2024): Half-year statutory income statement for Jan-June period."
    },
    "balance_sheet_2025_6m": {
        "df": df_2025_balance_sheet,
        "comment": "Consolidated Statement of Financial Position (6M 2025): Snapshot of assets, liabilities, and equity as of June 30, 2025."
    },
    "cash_flow_2025_6m": {
        "df": df_2025_cash_flow,
        "comment": "Consolidated Statement of Cash Flows (6M 2025/2024): Half-year breakdown of cash movements."
    },
    "core_ifrs_2025_6m": {
        "df": df_2025_core_ifrs,
        "comment": "CORE and IFRS Profit or Loss (6M 2025/2024): Half-year side-by-side comparison of IFRS vs CORE metrics."
    },
    "equity_free_cash_flow_2025_6m": {
        "df": df_2025_equity_free_cash_flow,
        "comment": "Equity Free Cash Flow (EFCF) (6M 2025/2024): Half-year calculation of cash available to shareholders."
    }
}

# 2. Loop through and save as Managed Tables
catalog = "sds"
schema = "cleaned"

for table_name, info in tables_to_deploy.items():
    full_table_path = f"{catalog}.{schema}.{table_name}"
    
    # 1. Convert Pandas to Spark and Save
    spark_df = spark.createDataFrame(info['df'])
    spark_df.write.mode("overwrite").saveAsTable(full_table_path)
    
    # 2. ESCAPE THE QUOTES: Replace ' with '' for SQL safety
    safe_comment = info['comment'].replace("'", "''")
    comment_sql = f"COMMENT ON TABLE {full_table_path} IS '{safe_comment}'"
    
    spark.sql(comment_sql)
    
    print(f"Deployed: {full_table_path}")

print("\nAll 10 tables are now live and up-to-date!")